In [2]:
!pip install ultralytics # installing ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 61.8 MB/s eta 0:00:00


In [3]:
from google.colab import drive
drive.mount('/content/drive',force_remount=True)

Mounted at /content/drive


In [4]:
from ultralytics import YOLO

model = YOLO("/content/drive/MyDrive/GP/yolo_experiments/Yolo26/overall_best26.pt")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


# Modified


In [5]:
import os
import numpy as np
import pandas as pd
import re
def process_test_sequence_fixed(folderN,folder, det_conf=0.25, closed_thr=0.5, open_thr=0.5,
                                min_blink_frames=3, gap_merge=2):
    files = sorted([f for f in os.listdir(folder) if f.endswith(('.jpg', '.png', '.jpeg'))])
#

#
    blink_results = []
    state_seq = []
    state_seq = [0] + state_seq
    closed_conf_seq = []
    closed_conf_seq = [0] + closed_conf_seq

    prev_state = 0  # assume open at start

    for f in files:
        img_path = os.path.join(folder, f)
        #
        if os.path.getsize(img_path) == 0:
          print("Skipped empty file:", img_path)
          continue
    #
        res = model.predict(img_path, conf=det_conf, verbose=False)

        closed_conf = 0.0
        open_conf = 0.0


        if res and len(res) > 0 and res[0].boxes is not None and len(res[0].boxes) > 0:
            for box in res[0].boxes:
                conf = float(box.conf[0])
                cls_id = int(box.cls[0])
                class_name = model.names[cls_id]

                if class_name == 'closed eyes':
                    closed_conf = max(closed_conf, conf)
                elif class_name == 'opened eyes':
                    open_conf = max(open_conf, conf)

        closed_conf_seq.append(closed_conf)
        frame_num = int(f.split('_')[-1].split('.')[0])
        index = len(closed_conf_seq) - 1


        # Stable frame-state decision
        if closed_conf >= closed_thr :# closed_conf >= open_conf cause model is bias towerd open
            state = 1
        elif open_conf >= open_thr and open_conf > closed_conf:
            state = 0
        else:
            state = 0 # carry previous state if uncertain / no detection

        state_seq.append(state)
        prev_state = state

    state_seq = np.array(state_seq, dtype=int)
    closed_conf_seq = np.array(closed_conf_seq, dtype=float)
    print("closed conf min/max:", min(closed_conf_seq), max(closed_conf_seq))
    print("first 50 closed confs:", closed_conf_seq[:50])#good
    # Remove tiny glitches: 0,1,0 or 1,0,1
    for i in range(1, len(state_seq) - 1):
        if state_seq[i - 1] == state_seq[i + 1]:
            state_seq[i] = state_seq[i - 1]

    # Extract closed runs
    raw_blinks = []
    in_blink = False

    si = -1
    print("num closed frames:", int(np.sum(state_seq)))

    for t in range(len(state_seq)):
        if not in_blink and state_seq[t] == 1:
            si = t
            in_blink = True
        elif in_blink and state_seq[t] == 0:
            ei = t - 1
            if ei >= si:
                raw_blinks.append((si, ei))
            in_blink = False #set them again
            si = -1

    if in_blink and si != -1:
        raw_blinks.append((si, len(state_seq) - 1))

    # Merge blink fragments separated by tiny open gaps
    merged = []
    for seg in raw_blinks:
        if not merged:
            merged.append(seg)
        else:
            prev_si, prev_ei = merged[-1]
            cur_si, cur_ei = seg
            if cur_si - prev_ei - 1 <= gap_merge:
                merged[-1] = (prev_si, cur_ei)
            else:
                merged.append(seg)

    print("Raw closed runs:")
    for si, ei in merged:
      print((si, ei), "duration =", ei - si + 1)


    # Keep only realistic blink durations AFTER trimming
    blinks = []
    for si, ei in merged:
        duration = ei - si + 1
        if min_blink_frames <= duration :
            blinks.append((si, ei))

    print("State sequence:")
    print(state_seq.tolist())
    print("\nDetected blinks:", blinks)

    if not blinks:
        print("No blink detected.")
        return

    maxi = 0
    for blink_idx, (si, ei) in enumerate(blinks):
        duration = ei - si + 1
        maxi = max(maxi, duration)

        # peak inside blink from closed confidence
        seg = closed_conf_seq[si:ei+1]
        bi = si + int(np.argmax(seg))

        baseline = min(closed_conf_seq[si], closed_conf_seq[ei])
        amplitude = closed_conf_seq[bi] - baseline

        if ei > bi:
            velocity = (closed_conf_seq[bi] - closed_conf_seq[ei]) / (ei - bi)
        else:
            velocity = 0.0#to avoid runtime error

        frequency = 100 * ((blink_idx + 1) / (ei + 1))#consider changing it
        print("start value:", closed_conf_seq[si])
        print("end value:", closed_conf_seq[ei])
        print(f"\n--- Blink {blink_idx + 1} ---")
        print(f"Interval B_i: [{si}, {ei}]")
        print(f"Duration: {duration} frames")
        print(f"baseline: {baseline} ")
        print(f"peak: {bi} ")
        print(f"Amplitude: {amplitude:.4f}")
        print(f"Reopening Velocity: {velocity:.4f}")
        print(f"Blink Frequency: {frequency:.2f} per 100 frames")
        print("first 50 closed confs:", closed_conf_seq[:50])

        blink_results.append({
            "Blink_ID": blink_idx + 1,
            "Start_Frame": si,
            "End_Frame": ei,
            "Duration": duration,
            "Amplitude": amplitude,
            "Velocity": velocity,
            "Frequency": frequency
        })
        df = pd.DataFrame(blink_results)
        base_path = '/content/drive/MyDrive/GP/CSV_completeDataset/valid'



        save_dir = os.path.join(base_path, label)
        os.makedirs(save_dir, exist_ok=True)

        csv_path = os.path.join(save_dir, f"{folderN}.csv")
        df.to_csv(csv_path, index=False)

        print("Saved:", csv_path)


**run all files here**

In [ ]:
import os

folder_path = '/content/drive/MyDrive/GP2/fps30_all_frames/val/Alert' #<-- Change this to your folder
label = "Alert"
image_extensions = (".jpg", ".jpeg", ".png", ".gif", ".bmp", ".tiff")

folders_with_images = []

# Step 1: collect folders
for root, dirs, files in os.walk(folder_path):
    if any(file.lower().endswith(image_extensions) for file in files):
        folders_with_images.append(root)

# Step 2: clean list
folders_with_images = sorted(set(folders_with_images))

print("Found folders:", len(folders_with_images))

# Step 3: process
for folder in folders_with_images:
    print("\nProcessing:", folder)

    folder_name = os.path.basename(folder)

    process_test_sequence_fixed(
        folder_name,folder,   # ✅ pass FULL PATH only
        det_conf=0.44,
        closed_thr=0.25,
        open_thr=0.25
    )

Streaming output truncated to the last 5000 lines.
Reopening Velocity: 0.1852
Blink Frequency: 2.42 per 100 frames
first 50 closed confs: [          0           0           0           0           0           0           0           0           0           0           0           0           0           0           0           0           0           0           0           0           0           0           0     0.85696     0.87828     0.85322
           0           0           0           0           0           0           0           0           0           0           0           0           0           0           0           0           0           0     0.77741     0.78002           0           0           0     0.87402]
Saved: /content/drive/MyDrive/GP/CSV_completeDataset/valid/Alert/A027_20260513_140624_frames.csv
start value: 0.641202986240387
end value: 0.8088923692703247

--- Blink 8 ---
Interval B_i: [322, 326]
Duration: 5 frames
baseline: 0.641202986240387 
peak: 323 


In [ ]:
import os
# if stopped use this
folder_path = '/content/drive/MyDrive/GP2/fps30_all_frames/val/Alert'
label = "Alert"
image_extensions = (".jpg", ".jpeg", ".png", ".gif", ".bmp", ".tiff")

start_folder = "/content/drive/MyDrive/GP2/fps30_all_frames/train/new alert/alert frame/A034_20260512_213325_frames"

folders_with_images = []

# Step 1: collect folders
for root, dirs, files in os.walk(folder_path):
    if any(file.lower().endswith(image_extensions) for file in files):
        folders_with_images.append(root)

folders_with_images = sorted(set(folders_with_images))

for f in folders_with_images:
    print(repr(f))

# Step 2: find where to start
if start_folder not in folders_with_images:
    print("⚠️ Start folder not found!")
else:
    start_idx = folders_with_images.index(start_folder)

    # Step 3: process from that point onward
    for folder in folders_with_images[start_idx:]:
        print("\nProcessing:", folder)

        folder_name = os.path.basename(folder)

        process_test_sequence_fixed(
            folder_name,folder,
            det_conf=0.44,
            closed_thr=0.25,
            open_thr=0.25
        )

'/content/drive/MyDrive/GP2/fps30_all_frames/train/new alert/alert frame/A001_20260512_140028_frames'
'/content/drive/MyDrive/GP2/fps30_all_frames/train/new alert/alert frame/A002_20260512_140407_frames'
'/content/drive/MyDrive/GP2/fps30_all_frames/train/new alert/alert frame/A003_20260512_140636_frames'
'/content/drive/MyDrive/GP2/fps30_all_frames/train/new alert/alert frame/A004_20260512_140924_frames'
'/content/drive/MyDrive/GP2/fps30_all_frames/train/new alert/alert frame/A005_20260512_141055_frames'
'/content/drive/MyDrive/GP2/fps30_all_frames/train/new alert/alert frame/A006_20260512_145924_frames'
'/content/drive/MyDrive/GP2/fps30_all_frames/train/new alert/alert frame/A007_20260512_150019_frames'
'/content/drive/MyDrive/GP2/fps30_all_frames/train/new alert/alert frame/A008_20260512_151408_frames'
'/content/drive/MyDrive/GP2/fps30_all_frames/train/new alert/alert frame/A009_20260512_151923_frames'
'/content/drive/MyDrive/GP2/fps30_all_frames/train/new alert/alert frame/A010_2026

In [ ]:
import os

# حطي هنا المجلد الكبير اللي فيه كل الفيديوهات
main_folder = "/content/drive/MyDrive/GP2/fps30_all_frames/train/new alert/alert frame"

image_extensions = (".jpg", ".jpeg", ".png", ".bmp", ".tiff")

video_folders = []

# يجمع كل الفولدرات اللي فيها صور
for root, dirs, files in os.walk(main_folder):

    image_files = [
        f for f in files
        if f.lower().endswith(image_extensions)
    ]

    if image_files:
        video_folders.append((root, len(image_files)))

# طباعة النتائج
print("Number of video folders:", len(video_folders))
print("-" * 50)

total_frames = 0

for folder, count in sorted(video_folders):

    folder_name = os.path.basename(folder)

    print(f"{folder_name} --> {count} frames")

    total_frames += count

print("-" * 50)
print("Total frames in all folders:", total_frames)

Number of video folders: 26
--------------------------------------------------
A001_20260512_140028_frames --> 6670 frames
A002_20260512_140407_frames --> 7490 frames
A003_20260512_140636_frames --> 2442 frames
A004_20260512_140924_frames --> 9146 frames
A005_20260512_141055_frames --> 5585 frames
A006_20260512_145924_frames --> 6416 frames
A007_20260512_150019_frames --> 4693 frames
A008_20260512_151408_frames --> 5501 frames
A009_20260512_151923_frames --> 5429 frames
A010_20260512_161125_frames --> 5501 frames
A011_20260512_161202_frames --> 3749 frames
A012_20260512_163318_frames --> 6984 frames
A013_20260512_163342_frames --> 7195 frames
A014_20260512_163419_frames --> 5613 frames
A015_20260512_163959_frames --> 7813 frames
A016_20260512_170821_frames --> 5707 frames
A017_20260512_173402_frames --> 5998 frames
A018_20260512_175300_frames --> 9195 frames
A019_20260512_175335_frames --> 7349 frames
A020_20260512_184500_frames --> 5506 frames
A021_20260512_201944_frames --> 5729 fram

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image


csv_path = "/content/drive/MyDrive/GP/CSV_completeDataset/training/Alert/A004_20260512_140924_frames.csv"

frames_folder = "/content/drive/MyDrive/GP2/fps30_all_frames/train/new alert/alert frame/A004_20260512_140924_frames"
# ======================

df = pd.read_csv(csv_path)

files = sorted([
    f for f in os.listdir(frames_folder)
    if f.lower().endswith(('.jpg', '.png', '.jpeg'))
])

print("Number of blinks:", len(df))

for idx, row in df.iterrows():

    start_f = int(row["Start_Frame"])
    end_f = int(row["End_Frame"])

    print(f"\n=== Blink {idx+1} ===")
    print(f"Frames: {start_f} -> {end_f}")

    for i in range(start_f, end_f + 1):


        img_path = os.path.join(frames_folder, files[i-1])

        img = Image.open(img_path)

        plt.figure(figsize=(10,10))
        plt.imshow(img)
        plt.axis("off")
        plt.title(f"Blink {idx+1} | Frame {i}")

        plt.show()

Output hidden; open in https://colab.research.google.com to view.